# Feature Review

Purpose:
- Verify feature table outputs
- Check train/validation/test splits
- Confirm no future leakage
- Inspect null rates
- Inspect feature distributions

In [1]:
import os
import sys
from pathlib import Path

import yaml
from pyspark.sql import functions as F

# Find project root automatically
project_root = Path.cwd()
while not (project_root / "configs").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
os.chdir(project_root)

Project root: /home/ubuntu/renewable-energy-forecasting-pipeline


In [2]:
PROJECT_USER_CONFIG = os.environ.get("PROJECT_USER_CONFIG")

with open(project_root / PROJECT_USER_CONFIG, "r") as f:
    user_config = yaml.safe_load(f)

In [3]:
from pyspark.sql import functions as F
from src.common.paths import paths
from src.common.spark_utils import get_spark_session

spark = get_spark_session("08-feature-review")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-83ded1b2-4a70-452e-ac0c-9166e48f8e83;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 361ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--------------------------------

In [4]:
feature_path = paths.gold_wind_ml_features
train_path = paths.gold_wind_ml_train
validation_path = paths.gold_wind_ml_validation
test_path = paths.gold_wind_ml_test

print(feature_path)
print(train_path)
print(validation_path)
print(test_path)

s3a://syed-datsbd-s2026/gold/wind/ml/features
s3a://syed-datsbd-s2026/gold/wind/ml/train
s3a://syed-datsbd-s2026/gold/wind/ml/validation
s3a://syed-datsbd-s2026/gold/wind/ml/test


In [5]:
features_df = spark.read.parquet(feature_path)
train_df = spark.read.parquet(train_path)
validation_df = spark.read.parquet(validation_path)
test_df = spark.read.parquet(test_path)

26/05/04 02:29:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
for name, df in {
    "features": features_df,
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    print(name, df.count(), "rows", len(df.columns), "columns")

features 537401 rows 54 columns


train 438286 rows 54 columns


validation 52608 rows 54 columns


test 46507 rows 54 columns


In [7]:
features_df.printSchema()

root
 |-- date_utc: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- day_of_year: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- daily_region_capacity_factor: double (nullable = true)
 |-- next_day_daily_region_capacity_factor: double (nullable = true)
 |-- mean_region_wind_speed_ms: double (nullable = true)
 |-- avg_station_wind_speed_std_ms: double (nullable = true)
 |-- daily_wind_speed_range_ms: double (nullable = true)
 |-- station_count: long (nullable = true)
 |-- total_hourly_observations: long (nullable = true)
 |-- is_low_wind_day: boolean (nullable = true)
 |-- is_high_wind_day: boolean (nullable = true)
 |-- cf_lag_1d: double (nullable = true)
 |-- cf_lag_2d: double (nullable = true)
 |-- cf_lag_3d: double (nullable = true)
 |-- cf_lag_7d: double (nullable = true)
 |-- cf_rolling_3d_mean: double (nullable 

In [8]:
features_df.select(
    F.min("date_utc").alias("min_date"),
    F.max("date_utc").alias("max_date"),
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|1995-01-01|2025-08-27|
+----------+----------+



In [9]:
features_df.groupBy("split").count().orderBy("split").show()

+----------+------+
|     split| count|
+----------+------+
|      test| 46507|
|     train|438286|
|validation| 52608|
+----------+------+



In [10]:
for name, df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    print(f"\n{name.upper()}")
    df.select(
        F.min("date_utc").alias("min_date"),
        F.max("date_utc").alias("max_date"),
        F.count("*").alias("rows"),
    ).show()


TRAIN


+----------+----------+------+
|  min_date|  max_date|  rows|
+----------+----------+------+
|1995-01-01|2019-12-31|438286|
+----------+----------+------+


VALIDATION


+----------+----------+-----+
|  min_date|  max_date| rows|
+----------+----------+-----+
|2020-01-01|2022-12-31|52608|
+----------+----------+-----+


TEST


+----------+----------+-----+
|  min_date|  max_date| rows|
+----------+----------+-----+
|2023-01-01|2025-08-27|46507|
+----------+----------+-----+



## Required Column Checks

In [11]:
required_cols = [
    "state",
    "date_utc",
    "daily_region_capacity_factor",
    "next_day_daily_region_capacity_factor",
    "daily_region_capacity_factor_lag_1d",
    "daily_region_capacity_factor_lag_7d",
    "daily_region_capacity_factor_rolling_7d_mean",
    "month",
    "season",
    "split",
]

missing = [c for c in required_cols if c not in features_df.columns]
missing

[]

In [12]:
assert missing == []

## Null Rate Review

In [13]:
null_check_cols = required_cols

features_df.select(
    [
        F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls")
        for c in null_check_cols
    ]
).show(truncate=False)

+-----------+--------------+----------------------------------+-------------------------------------------+-----------------------------------------+-----------------------------------------+--------------------------------------------------+-----------+------------+-----------+
|state_nulls|date_utc_nulls|daily_region_capacity_factor_nulls|next_day_daily_region_capacity_factor_nulls|daily_region_capacity_factor_lag_1d_nulls|daily_region_capacity_factor_lag_7d_nulls|daily_region_capacity_factor_rolling_7d_mean_nulls|month_nulls|season_nulls|split_nulls|
+-----------+--------------+----------------------------------+-------------------------------------------+-----------------------------------------+-----------------------------------------+--------------------------------------------------+-----------+------------+-----------+
|0          |0             |0                                 |0                                          |48                                       |336        

In [14]:
features_df.select(
    [
        (F.sum(F.col(c).isNull().cast("int")) / F.count("*")).alias(f"{c}_null_rate")
        for c in null_check_cols
    ]
).show(truncate=False)

+---------------+------------------+--------------------------------------+-----------------------------------------------+---------------------------------------------+---------------------------------------------+------------------------------------------------------+---------------+----------------+---------------+
|state_null_rate|date_utc_null_rate|daily_region_capacity_factor_null_rate|next_day_daily_region_capacity_factor_null_rate|daily_region_capacity_factor_lag_1d_null_rate|daily_region_capacity_factor_lag_7d_null_rate|daily_region_capacity_factor_rolling_7d_mean_null_rate|month_null_rate|season_null_rate|split_null_rate|
+---------------+------------------+--------------------------------------+-----------------------------------------------+---------------------------------------------+---------------------------------------------+------------------------------------------------------+---------------+----------------+---------------+
|0.0            |0.0               |0.0 

## Target Validation

In [15]:
features_df.select(
    F.min("next_day_daily_region_capacity_factor").alias("target_min"),
    F.max("next_day_daily_region_capacity_factor").alias("target_max"),
    F.avg("next_day_daily_region_capacity_factor").alias("target_mean"),
).show()

+----------+----------+--------------------+
|target_min|target_max|         target_mean|
+----------+----------+--------------------+
|       0.0|  0.900287|0.042603142143390386|
+----------+----------+--------------------+



In [16]:
invalid_target_count = features_df.filter(
    (F.col("next_day_daily_region_capacity_factor") < 0)
    | (F.col("next_day_daily_region_capacity_factor") > 1)
    | F.col("next_day_daily_region_capacity_factor").isNull()
).count()

invalid_target_count

0

In [17]:
assert invalid_target_count == 0

## Leakage Check: Lag Alignment

For a given state and date, `lag_1d` should equal the previous day's `daily_region_capacity_factor`.

In [18]:
sample_state = "TX"

(
    features_df
    .filter(F.col("state") == sample_state)
    .select(
        "state",
        "date_utc",
        "daily_region_capacity_factor",
        "next_day_daily_region_capacity_factor",
        "daily_region_capacity_factor_lag_1d",
        "daily_region_capacity_factor_rolling_7d_mean",
        "split",
    )
    .orderBy("date_utc")
    .show(20, truncate=False)
)

+-----+----------+----------------------------+-------------------------------------+-----------------------------------+--------------------------------------------+-----+
|state|date_utc  |daily_region_capacity_factor|next_day_daily_region_capacity_factor|daily_region_capacity_factor_lag_1d|daily_region_capacity_factor_rolling_7d_mean|split|
+-----+----------+----------------------------+-------------------------------------+-----------------------------------+--------------------------------------------+-----+
|TX   |1995-01-01|0.082865                    |0.030617                             |NULL                               |NULL                                        |train|
|TX   |1995-01-02|0.030617                    |0.037708                             |0.082865                           |0.082865                                    |train|
|TX   |1995-01-03|0.037708                    |0.072887                             |0.030617                           |0.056741      

In [19]:
from pyspark.sql.window import Window

w = Window.partitionBy("state").orderBy("date_utc")

leakage_check_df = (
    features_df
    .withColumn(
        "expected_lag_1d",
        F.lag("daily_region_capacity_factor", 1).over(w),
    )
    .withColumn(
        "lag_1d_diff",
        F.abs(
            F.col("daily_region_capacity_factor_lag_1d")
            - F.col("expected_lag_1d")
        ),
    )
)

bad_lag_rows = leakage_check_df.filter(
    F.col("lag_1d_diff") > 1e-9
).count()

bad_lag_rows

0

In [20]:
assert bad_lag_rows == 0

## Split Boundary Leakage Check

In [21]:
split_ranges = (
    features_df
    .groupBy("split")
    .agg(
        F.min("date_utc").alias("min_date"),
        F.max("date_utc").alias("max_date"),
        F.count("*").alias("rows"),
    )
    .orderBy("min_date")
)

split_ranges.show(truncate=False)

+----------+----------+----------+------+
|split     |min_date  |max_date  |rows  |
+----------+----------+----------+------+
|train     |1995-01-01|2019-12-31|438286|
|validation|2020-01-01|2022-12-31|52608 |
|test      |2023-01-01|2025-08-27|46507 |
+----------+----------+----------+------+



In [22]:
train_bad = train_df.filter(F.col("date_utc") > F.lit("2019-12-31")).count()
validation_bad = validation_df.filter(
    (F.col("date_utc") < F.lit("2020-01-01"))
    | (F.col("date_utc") > F.lit("2022-12-31"))
).count()
test_bad = test_df.filter(F.col("date_utc") < F.lit("2023-01-01")).count()

train_bad, validation_bad, test_bad

(0, 0, 0)

In [23]:
assert train_bad == 0
assert validation_bad == 0
assert test_bad == 0

## Feature Distribution Summary

In [24]:
numeric_review_cols = [
    "daily_region_capacity_factor",
    "next_day_daily_region_capacity_factor",
    "daily_region_capacity_factor_lag_1d",
    "daily_region_capacity_factor_lag_7d",
    "daily_region_capacity_factor_rolling_7d_mean",
    "mean_region_wind_speed_ms",
    "daily_wind_speed_range_ms",
]

features_df.select(numeric_review_cols).summary().show(truncate=False)

26/05/04 02:33:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+----------------------------+-------------------------------------+-----------------------------------+-----------------------------------+--------------------------------------------+-------------------------+-------------------------+
|summary|daily_region_capacity_factor|next_day_daily_region_capacity_factor|daily_region_capacity_factor_lag_1d|daily_region_capacity_factor_lag_7d|daily_region_capacity_factor_rolling_7d_mean|mean_region_wind_speed_ms|daily_wind_speed_range_ms|
+-------+----------------------------+-------------------------------------+-----------------------------------+-----------------------------------+--------------------------------------------+-------------------------+-------------------------+
|count  |537401                      |537401                               |537353                             |537065                             |537353                                      |537401                   |537401                   |
|mean   |0.04260

In [25]:
features_df.groupBy("season").count().orderBy("season").show()

+------+------+
|season| count|
+------+------+
|  fall|131040|
|spring|136895|
|summer|136651|
|winter|132815|
+------+------+



In [26]:
features_df.groupBy("state").count().orderBy("state").show(60, truncate=False)

+-----+-----+
|state|count|
+-----+-----+
|AL   |11197|
|AR   |11196|
|AZ   |11196|
|CA   |11196|
|CO   |11196|
|CT   |11195|
|DE   |11193|
|FL   |11196|
|GA   |11196|
|IA   |11196|
|ID   |11196|
|IL   |11196|
|IN   |11196|
|KS   |11196|
|KY   |11196|
|LA   |11196|
|MA   |11195|
|MD   |11195|
|ME   |11196|
|MI   |11196|
|MN   |11196|
|MO   |11196|
|MS   |11196|
|MT   |11196|
|NC   |11196|
|ND   |11196|
|NE   |11196|
|NH   |11196|
|NJ   |11195|
|NM   |11196|
|NV   |11196|
|NY   |11196|
|OH   |11196|
|OK   |11196|
|OR   |11196|
|PA   |11196|
|RI   |11196|
|SC   |11196|
|SD   |11196|
|TN   |11196|
|TX   |11196|
|UT   |11196|
|VA   |11196|
|VT   |11195|
|WA   |11196|
|WI   |11196|
|WV   |11196|
|WY   |11196|
+-----+-----+



## Split-Level Target Summary

In [27]:
features_df.groupBy("split").agg(
    F.count("*").alias("rows"),
    F.avg("next_day_daily_region_capacity_factor").alias("target_mean"),
    F.stddev("next_day_daily_region_capacity_factor").alias("target_stddev"),
    F.min("next_day_daily_region_capacity_factor").alias("target_min"),
    F.max("next_day_daily_region_capacity_factor").alias("target_max"),
).orderBy("split").show(truncate=False)

+----------+------+--------------------+--------------------+----------+----------+
|split     |rows  |target_mean         |target_stddev       |target_min|target_max|
+----------+------+--------------------+--------------------+----------+----------+
|test      |46507 |0.03896842815060113 |0.051982736377265026|0.0       |0.778695  |
|train     |438286|0.04304203873269997 |0.056724096621248346|0.0       |0.900287  |
|validation|52608 |0.042159814381842964|0.05807126685112137 |0.0       |0.891498  |
+----------+------+--------------------+--------------------+----------+----------+



## Final Validation Status

The feature table is considered trusted if:

- Required columns exist
- Target has no nulls
- Target remains within [0, 1]
- Lag features align with past values
- Rolling features exclude current-day values
- Splits are strictly time-based
- Train, validation, and test tables are non-empty

In [28]:
assert features_df.count() > 0
assert train_df.count() > 0
assert validation_df.count() > 0
assert test_df.count() > 0

print("Feature review passed.")

Feature review passed.
